In [ ]:
#!sudo apt-get update -y
#!sudo apt-get install python3.10 python3.10-distutils -y

#!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
#!sudo update-alternatives --set python3 /usr/bin/python3.10

#!curl -sS https://bootstrap.pypa.io/get-pip.py | python3

# SETUP

In [ ]:
#!pip3 uninstall --yes tensorflow jax jaxlib torch torch_xla torchvision torchaudio torchtext torchdata ml-dtypes
#!pip3 cache purge

#!pip3 install torch~=2.6.0 'torch_xla[tpu]~=2.6.0' torchvision torchaudio \
#  -f https://storage.googleapis.com/libtpu-releases/index.html \
#  -f https://storage.googleapis.com/libtpu-wheels/index.html
#!pip3 install -U "jax[tpu]"

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
import torch_xla.core.xla_model as xm
import time

In [2]:
print(jax.devices())
key = jax.random.PRNGKey(0)
x = jax.random.normal(key, (1000, 1000))
y = jnp.dot(x, x.T)
print(y.block_until_ready())

device = xm.xla_device()
print(device)
a = torch.randn(3, 3).to(xm.xla_device())
b = torch.randn(3, 3).to(xm.xla_device())
c = a @ b
#print(c)

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(0,0,0), core_on_chip=1), TpuDevice(id=2, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,0,0), core_on_chip=1), TpuDevice(id=4, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(0,1,0), core_on_chip=1), TpuDevice(id=6, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,1,0), core_on_chip=1)]
[[ 9.68059326e+02 -2.70215626e+01  2.63414707e+01 ...  6.40200329e+00
   4.52859879e-01  1.05772581e+01]
 [-2.70215626e+01  1.04650012e+03  3.69840355e+01 ... -6.26201820e+00
   1.33741150e+01 -3.40158386e+01]
 [ 2.63414707e+01  3.69840355e+01  1.04029211e+03 ...  6.80434723e+01
   7.29222584e+00  7.43697968e+01]
 ...
 [ 6.40200329e+00 -6.26201820e+00  6.80434723e+01 ...  9.76772034e+02
   1.56926460e+01 -4.04213409e+01]
 [ 4.52859879e-01  1.33741150e+01  7.29222584e

# COMPARE JITs

In [3]:
# input tensors
A_np = np.random.rand(100, 100)
B_np = np.random.rand(100, 100)

A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)
A_torch = torch.from_numpy(A_np)
B_torch = torch.from_numpy(B_np)



# functions for native JIT
def jax_function(A, B):
	C = B
	for _ in range(50):
		C = jnp.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
		C = jnp.linalg.inv(C)
	d = jnp.linalg.det(C)
	return d

def pyt_function(A, B):
	C = B
	for _ in range(50):
		C = torch.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = C.T
		C = torch.linalg.inv(C)
	d = torch.linalg.det(C)
	return d

# trace & warumup for native JIT
jax_function_jit = jax.jit(jax_function)
pyt_function_jit = torch.jit.script(pyt_function)
for _ in range(100): yt = jax_function_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_jit(A_torch, B_torch)

In [4]:
# functions for JIT+XLA
def jax_function_alt(A, B):
	C = B
	for _ in range(50):
		C = jnp.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
	return jnp.sum(C)

def pyt_function_alt(A, B):
	C = B
	for _ in range(50):
		C = torch.matmul(A, C)
		C = (C + A + B) / 3
		C = C * A * B
		C = C.T
	return torch.sum(C)

# trace & warm up for JIT+XLA
device = xm.xla_device()
A_torch_xla = A_torch.to(device)
B_torch_xla = B_torch.to(device)

jax_function_alt_jit = jax.jit(jax_function_alt)
pyt_function_alt_jit = torch.jit.script(pyt_function_alt)
for _ in range(100): yt = jax_function_alt_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_alt_jit(A_torch_xla, B_torch_xla)

In [ ]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"JAX avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time: 0.028981 seconds
PyT avg time: 0.355651 seconds


In [ ]:
# native JIT (XLA for JAX, TorchScript for PyTorch)

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JAX avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch, B_torch)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

JAX avg time (XLA): 0.006901 seconds
PyT avg time  (TS): 0.308116 seconds


In [ ]:
# JIT + XLA

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_alt_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_alt_jit(A_torch_xla, B_torch_xla)
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

Jax avg time (XLA): 0.001128 seconds
PyT avg time (XLA): 0.003007 seconds


# COMPARE AUTOGRADS

In [5]:
# grad, trace & warm up for native JIT
jax_function_grad = jax.grad(jax_function)
A_torch_grad = A_torch.requires_grad_(True)
B_torch_grad = B_torch.requires_grad_(True)

jax_function_grad_jit = jax.jit(jax_function_grad)
for _ in range(100): yt = jax_function_grad_jit(A_jax, B_jax).block_until_ready()
for _ in range(100): yt = pyt_function_jit(A_torch_grad, B_torch_grad).backward()

In [6]:
# grad, trace & warm up for JIT+XLA
jax_function_alt_grad = jax.grad(jax_function_alt)

device = xm.xla_device()
A_torch_grad_xla = A_torch_grad.to(device)
B_torch_grad_xla = B_torch_grad.to(device)

jax_function_alt_grad_jit = jax.jit(jax_function_alt_grad)
for _ in range(100): yt = jax_function_alt_grad_jit(A_jax, B_jax).block_until_ready()
#for _ in range(100): yt = pyt_function_alt_jit(B_torch_grad_xla, B_torch_grad_xla).backward()

In [7]:
# Autograd native JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_jit(A_torch_grad, B_torch_grad).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")

Jax avg time (XLA): 0.004351 seconds
PyT avg time  (TS): 0.689095 seconds


In [8]:
# Autograd JIT+XLA

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_alt_grad_jit(A_jax, B_jax)
	end = time.time()
	runtimes.append(end - start)
print(f"Jax avg time (XLA): {sum(runtimes)/len(runtimes):.6f} seconds")

runtimes = list()
for _ in range(100):
	start = time.time()
	y = pyt_function_alt_jit(A_torch_grad_xla, B_torch_grad_xla).backward()
	end = time.time()
	runtimes.append(end - start)
print(f"PyT avg time  (TS): {sum(runtimes)/len(runtimes):.6f} seconds")


Jax avg time (XLA): 0.000220 seconds
